![Week 4 Day 4](../../assets/banners/w4d4.png)

# Week 4 Day 4 - Solution: Being the Mapper's Adversary

Two parts: hunt for label styles `normalise_label` gets wrong, deciding for each whether refusing is the safer behaviour; then fake a three-page Q4(b) with a crossed-out paragraph and check `map_answers`.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Try the exercise yourself before reading on: struggling a little is where the learning happens! These are <i>reference</i> solutions, not the only right answers.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Most of today's cells still run offline; the few that call a model will")
        print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
from gradewise.config import DATA_DIR

In [ ]:
from gradewise import datasets
from gradewise.schemas import ExamPaper

EXAM_FILE = DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json"
exam = ExamPaper.model_validate_json(EXAM_FILE.read_text(encoding="utf-8"))
QUESTION_IDS = [q.id for q in exam.questions]  # 💡 the mapper always needs every ID on the paper
# 📦 The questions we SCORE: all of them, or Q1–Q3 (all their parts) with DATA_SIZE=small
EVAL_IDS = [qid for qid in QUESTION_IDS if qid in set(datasets.exam_question_ids())]
print(f"{exam.title}: {len(QUESTION_IDS)} questions → {', '.join(QUESTION_IDS)}")
print(f"Scored in this run: {', '.join(EVAL_IDS)}")

In [ ]:
def show_table(rows: list[dict]) -> None:
    """Render a list of dicts as a Markdown table (a tiny stand-in for pandas: no extra dependencies)."""
    if not rows:
        print("(no rows)")
        return
    cols = list(rows[0])

    def cell(value) -> str:
        return str(value).replace("|", "\\|").replace("\n", " ⏎ ")

    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(cell(r.get(c, "")) for c in cols) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

### Part 1: tricky labels

In [ ]:
from gradewise.transcription import normalise_label

labels = {
    "Q 4 - b": "accept: dashes are just separators",
    "4.b.": "accept: dots are just separators",
    "Q4B": "accept: case doesn't matter",
    "No. 4(b)": "accept: 'No.' is a word prefix",
    "Part b of 4": "refuse is fine: word order reversed, rare, and a human can place it",
    "IV(b)": "refuse is SAFER: roman question numbers are ambiguous (IV vs 4 vs a stray mark)",
    "4(b)(i)": "accept: no 4(b)(i) on this exam, so keep the deepest real ID; it can only be part of 4(b)",
    "Answer to 4b": "accept: 'Answer to' / 'Answer for' are just wordy prefixes",
    "(a)  [inside Q1]": "refuse is SAFER: Q1 has no parts, so this could belong to any question's (a)",
}
show_table(
    [
        {
            "label": repr(label),
            "→": normalise_label(label.split("  [")[0], QUESTION_IDS, current="Q1"),
            "should it?": why,
        }
        for label, why in labels.items()
    ]
)

💡 The guiding principle: **a refusal is cheap, a wrong match is expensive.** A refused label lands in `unmapped_text`, lowers the confidence of blank answers to 0.5, and a human places it in seconds. A wrong match silently grades the text against the wrong question. That's exactly how GradeWise's mapper evolved: `4(b)(i)` → `Q4b` and "Answer to 4b" → `Q4b` were added because they are frequent **and** unambiguous, while a bare `(a)` under a question with no parts is now refused and sent to `unmapped_text`. Extend it the same way: one unambiguous pattern at a time, each with a test in `tests/test_transcription.py`.

### Part 2: one answer across three pages, with a crossing-out

In [ ]:
from gradewise.transcription import AnswerSegment, PageTranscription, map_answers

pages = [
    PageTranscription(
        page_number=1,
        confidence=0.9,
        segments=[
            AnswerSegment(label="Q4(a)", text="Two threads change shared data at once.", confidence=0.95),
            AnswerSegment(label="4 (b)", text="A semafore is an integer variable", confidence=0.9),
        ],
    ),
    PageTranscription(
        page_number=2,
        confidence=0.85,
        segments=[
            AnswerSegment(
                label=None,
                continued=True,
                confidence=0.85,
                text="with two atomic operations, wait() and signal().",
                crossed_out=["A semaphore is the same as a mutex lock so it can only be 0 or 1."],
            ),
            AnswerSegment(label="Q5", text="offset = 13 bits", confidence=0.95),
        ],
    ),
    PageTranscription(
        page_number=3,
        confidence=0.9,
        segments=[
            AnswerSegment(
                label="Q4b contd.", text="wait() decrements and blocks, signal() increments and wakes.", confidence=0.9
            ),
        ],
    ),
]
sheet = map_answers(pages, exam, "sam")
q4b = sheet.answer_for("Q4b")
print("text:\n ", q4b.text.replace("\n", "\n  "))
print("pages:", q4b.pages, "· crossed_out:", q4b.crossed_out, "· confidence:", q4b.confidence)

assert q4b.pages == [1, 2, 3]
assert q4b.text.startswith("A semafore is an integer variable")  # 💡 spelling untouched
assert "wait() decrements" in q4b.text and "mutex lock" not in q4b.text
assert q4b.crossed_out == ["A semaphore is the same as a mutex lock so it can only be 0 or 1."]
assert sheet.answer_for("Q5").text == "offset = 13 bits"  # Q5 sat between the two halves, untouched
print("\n✅ all checks passed")

Three things made this work: the unlabelled segment at the top of page 2 continued the question we were *in* (Q4b, not Q4a); "Q4b contd." on page 3 **reopened** Q4b even though Q5 came in between; and the crossed-out paragraph never reached `text`, so it can't be marked, but it's kept in `crossed_out` in case a human disagrees.

🧪 **TRY:** turn these asserts into a real test function in `tests/test_transcription.py` style. That's how every rule in `mapper.py` is protected.